In [1]:
!pip install nltk # Menginstal pustaka NLTK  jika belum terpasang dalam sistem

In [2]:
import nltk # Mengimpor pustaka NLTK untuk pemrosesan bahasa alami
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import numpy as np
import pandas as pd
import re
import math
from tqdm import tqdm

nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
stop_words = set(stopwords.words('indonesian')) # Changed 'indonesia' to 'indonesian'

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


# https://drive.google.com/file/d/1ysv88LcYDyjYrSrC_Fke4JThiv2BuP9y/view?usp=drive_link

In [3]:
!pip install gdown
import gdown

file_id = "1ysv88LcYDyjYrSrC_Fke4JThiv2BuP9y"
url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(url, "data.xml", quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1ysv88LcYDyjYrSrC_Fke4JThiv2BuP9y
From (redirected): https://drive.google.com/uc?id=1ysv88LcYDyjYrSrC_Fke4JThiv2BuP9y&confirm=t&uuid=7dc2635c-6615-40f8-9e89-066b1a40b80d
To: /content/data.xml
100%|██████████| 2.56M/2.56M [00:00<00:00, 32.8MB/s]


'data.xml'

In [4]:
df = pd.read_xml("data.xml")
df.head()

,id,sumber,tanggal,kategori,judul,isi,link,jumlahkata
0,1,kompas.com,2015/07/01,Teknologi,"Ponsel Huawei Honor 4C Dibanderol Rp 2,2 Juta","JAKARTA, KOMPAS.com Ponsel Android Huawei Hon...",http://tekno.kompas.com/read/xml/2015/07/01/18...,315
1,2,kompas.com,2015/07/01,Teknologi,Asosiasi: RPP E-commerce Tidak Sesuai Hasil Di...,"JAKARTA, KOMPAS.com - Sejak 2013, wacana tenta...",http://tekno.kompas.com/read/xml/2015/07/01/16...,419
2,3,kompas.com,2015/07/01,Teknologi,"Pemesan ""iPhone Jadi Sabun"" Karyawan Pesaing L...","JAKARTA, KOMPAS.com Danis Darusman, pelanggan ...",http://tekno.kompas.com/read/xml/2015/07/01/15...,265
3,4,kompas.com,2015/07/01,Teknologi,"""Autofeather Failure"", Momok bagi Pesawat Bali...",KOMPAS.com Salah satu momok yang dihadapi dala...,http://tekno.kompas.com/read/xml/2015/07/01/14...,481
4,5,kompas.com,2015/07/01,Teknologi,Laptop Bezel Tertipis di Dunia Masuk Indonesia,"JAKARTA, KOMPAS.com - Resmi diperkenalkan pada...",http://tekno.kompas.com/read/xml/2015/07/01/14...,227


In [5]:
df.tail()

,id,sumber,tanggal,kategori,judul,isi,link,jumlahkata
1497,1498,tribunnews.com,2015/07/01,Travel,Kuliner Medan Sedap Harga Murah Meriah: Bubur ...,"TRIBUNNEWS.COM - Selain Bika Ambon, tiga kulin...",http://www.tribunnews.com/travel/2015/07/01/ku...,101
1498,1499,tribunnews.com,2015/07/01,Travel,Seribu Delegasi 60 Negara Akan Ikuti Pameran W...,TRIBUNNEWS.COM - Asosiasi nirlaba internasiona...,http://www.tribunnews.com/travel/2015/07/01/se...,197
1499,1500,tribunnews.com,2015/07/01,Travel,Bebek Goreng Gurihnya Mantap Saat Disantap Den...,"Laporan wartawan Tribun Medan, Silfa Humairah ...",http://www.tribunnews.com/travel/2015/07/01/be...,125
1500,1501,tribunnews.com,2015/07/01,Travel,Kisah Haru Penjual Siomay Pink setelah Berulan...,"Laporan Wartawan Tribunnews, Reynas Abdila TRI...",http://www.tribunnews.com/travel/2015/07/01/ki...,95
1501,1502,tribunnews.com,2015/07/01,Travel,"Nanang Tolak Tawaran Rp 2,5 Miliar untuk Fosil...","Laporan Wartawan Pos Kupang, Muhlis Al Alawi T...",http://www.tribunnews.com/travel/2015/07/01/na...,148


In [6]:
df.drop(columns=['sumber', 'tanggal', 'link'], inplace=True)
df.head()

,id,kategori,judul,isi,jumlahkata
0,1,Teknologi,"Ponsel Huawei Honor 4C Dibanderol Rp 2,2 Juta","JAKARTA, KOMPAS.com Ponsel Android Huawei Hon...",315
1,2,Teknologi,Asosiasi: RPP E-commerce Tidak Sesuai Hasil Di...,"JAKARTA, KOMPAS.com - Sejak 2013, wacana tenta...",419
2,3,Teknologi,"Pemesan ""iPhone Jadi Sabun"" Karyawan Pesaing L...","JAKARTA, KOMPAS.com Danis Darusman, pelanggan ...",265
3,4,Teknologi,"""Autofeather Failure"", Momok bagi Pesawat Bali...",KOMPAS.com Salah satu momok yang dihadapi dala...,481
4,5,Teknologi,Laptop Bezel Tertipis di Dunia Masuk Indonesia,"JAKARTA, KOMPAS.com - Resmi diperkenalkan pada...",227


In [7]:
df['kategori'].unique()

array(['Teknologi', 'Bisnis Ekonomi', 'Nasional', 'Olahraga', 'Travel',
       'Bola', 'Otomotif', 'Lifestyle'], dtype=object)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1502 entries, 0 to 1501
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          1502 non-null   int64 
 1   kategori    1502 non-null   object
 2   judul       1502 non-null   object
 3   isi         1502 non-null   object
 4   jumlahkata  1502 non-null   int64 
dtypes: int64(2), object(3)
memory usage: 58.8+ KB


In [9]:
df.isnull().sum()

,0
id,0
kategori,0
judul,0
isi,0
jumlahkata,0


In [10]:
df.duplicated().sum()

np.int64(0)

# Tahap Preprocessing Data

In [11]:
def caseFolding(text):
  text = text.lower()
  return text


contoh = "Terima Kasih, Kak! Kamu sangat baik sekali kepadaku hari ini."
print(f'original: {contoh}')
print(f'case folded: {caseFolding(contoh)}')

original: Terima Kasih, Kak! Kamu sangat baik sekali kepadaku hari ini.
case folded: terima kasih, kak! kamu sangat baik sekali kepadaku hari ini.


In [12]:
def punctuationRemoval(text):
  text = re.sub(r'[^\w\s]', '', text)
  return text


contoh = "Terima kasih, Kak! Kamu sangat baik sekali kepadaku hari ini."
print(f'original: {contoh}')
print(f'punction removed: {punctuationRemoval(contoh)}')

original: Terima kasih, Kak! Kamu sangat baik sekali kepadaku hari ini.
punction removed: Terima kasih Kak Kamu sangat baik sekali kepadaku hari ini


In [13]:
stop_words

{'ada',
 'adalah',
 'adanya',
 'adapun',
 'agak',
 'agaknya',
 'agar',
 'akan',
 'akankah',
 'akhir',
 'akhiri',
 'akhirnya',
 'aku',
 'akulah',
 'amat',
 'amatlah',
 'anda',
 'andalah',
 'antar',
 'antara',
 'antaranya',
 'apa',
 'apaan',
 'apabila',
 'apakah',
 'apalagi',
 'apatah',
 'artinya',
 'asal',
 'asalkan',
 'atas',
 'atau',
 'ataukah',
 'ataupun',
 'awal',
 'awalnya',
 'bagai',
 'bagaikan',
 'bagaimana',
 'bagaimanakah',
 'bagaimanapun',
 'bagi',
 'bagian',
 'bahkan',
 'bahwa',
 'bahwasanya',
 'baik',
 'bakal',
 'bakalan',
 'balik',
 'banyak',
 'bapak',
 'baru',
 'bawah',
 'beberapa',
 'begini',
 'beginian',
 'beginikah',
 'beginilah',
 'begitu',
 'begitukah',
 'begitulah',
 'begitupun',
 'bekerja',
 'belakang',
 'belakangan',
 'belum',
 'belumlah',
 'benar',
 'benarkah',
 'benarlah',
 'berada',
 'berakhir',
 'berakhirlah',
 'berakhirnya',
 'berapa',
 'berapakah',
 'berapalah',
 'berapapun',
 'berarti',
 'berawal',
 'berbagai',
 'berdatangan',
 'beri',
 'berikan',
 'berikut'

In [14]:
def remove_stopwords(text):
  tokens = text.split()
  filtered = [] # Initialize as an empty list

  # Loop setiap kata
  for word in tokens:
    if word not in stop_words:
      filtered.append(word)

  # Menggabungkan kembali kata- kata menjadi kalimat
  cleaned_text = " ".join(filtered)
  return cleaned_text

contoh_teks = "saya aslinya adalah duyung yang tidak suka berenang"
print("Teks asli     :", contoh_teks)
print("Setelah proses:", remove_stopwords(contoh_teks))

Teks asli     : saya aslinya adalah duyung yang tidak suka berenang
Setelah proses: aslinya duyung suka berenang


In [15]:
df['clean'] = df['isi'].apply(caseFolding)
df['clean'] = df['clean'].apply(punctuationRemoval)
df['clean'] = df['clean'].apply(remove_stopwords)

df.head()

,id,kategori,judul,isi,jumlahkata,clean
0,1,Teknologi,"Ponsel Huawei Honor 4C Dibanderol Rp 2,2 Juta","JAKARTA, KOMPAS.com Ponsel Android Huawei Hon...",315,jakarta kompascom ponsel android huawei honor ...
1,2,Teknologi,Asosiasi: RPP E-commerce Tidak Sesuai Hasil Di...,"JAKARTA, KOMPAS.com - Sejak 2013, wacana tenta...",419,jakarta kompascom 2013 wacana rancangan peratu...
2,3,Teknologi,"Pemesan ""iPhone Jadi Sabun"" Karyawan Pesaing L...","JAKARTA, KOMPAS.com Danis Darusman, pelanggan ...",265,jakarta kompascom danis darusman pelanggan laz...
3,4,Teknologi,"""Autofeather Failure"", Momok bagi Pesawat Bali...",KOMPAS.com Salah satu momok yang dihadapi dala...,481,kompascom salah momok dihadapi pesawat turbopr...
4,5,Teknologi,Laptop Bezel Tertipis di Dunia Masuk Indonesia,"JAKARTA, KOMPAS.com - Resmi diperkenalkan pada...",227,jakarta kompascom resmi diperkenalkan ajang co...


# Tahap TF

TF = ukuran seberapa sering kata muncul dalam satu dokumen.

TF(t,d)= jumlah kemunculan kata t dalam dokumen d

In [16]:
def compute_word_frequency(text):
  words = word_tokenize(text)
  frequency_table = {}

  for word in words:
    if word in frequency_table:
      frequency_table[word] += 1
    else:
      frequency_table[word] = 1

  return frequency_table

In [17]:
df['word_frequency'] = df['clean'].apply(compute_word_frequency)
df.head()

,id,kategori,judul,isi,jumlahkata,clean,word_frequency
0,1,Teknologi,"Ponsel Huawei Honor 4C Dibanderol Rp 2,2 Juta","JAKARTA, KOMPAS.com Ponsel Android Huawei Hon...",315,jakarta kompascom ponsel android huawei honor ...,"{'jakarta': 1, 'kompascom': 1, 'ponsel': 5, 'a..."
1,2,Teknologi,Asosiasi: RPP E-commerce Tidak Sesuai Hasil Di...,"JAKARTA, KOMPAS.com - Sejak 2013, wacana tenta...",419,jakarta kompascom 2013 wacana rancangan peratu...,"{'jakarta': 2, 'kompascom': 1, '2013': 1, 'wac..."
2,3,Teknologi,"Pemesan ""iPhone Jadi Sabun"" Karyawan Pesaing L...","JAKARTA, KOMPAS.com Danis Darusman, pelanggan ...",265,jakarta kompascom danis darusman pelanggan laz...,"{'jakarta': 1, 'kompascom': 1, 'danis': 12, 'd..."
3,4,Teknologi,"""Autofeather Failure"", Momok bagi Pesawat Bali...",KOMPAS.com Salah satu momok yang dihadapi dala...,481,kompascom salah momok dihadapi pesawat turbopr...,"{'kompascom': 1, 'salah': 2, 'momok': 1, 'diha..."
4,5,Teknologi,Laptop Bezel Tertipis di Dunia Masuk Indonesia,"JAKARTA, KOMPAS.com - Resmi diperkenalkan pada...",227,jakarta kompascom resmi diperkenalkan ajang co...,"{'jakarta': 2, 'kompascom': 1, 'resmi': 1, 'di..."


# Tahap IDF

In [18]:
# Jumlah total dokumen
N = len(df)

# Menghitung Document Frequency (dari banyak dokumen mengandung term tertentu)
document_frequencies = {}

for word_freq in df['word_frequency']:
  for word in word_freq.keys():
    if word in document_frequencies:
      document_frequencies[word] +=1
    else:
      document_frequencies[word] =1

# Menyimpan kedalam bentuk dictionary
idf = {}

# Menghitung IDF untuk setiap kata
for word, df_t in document_frequencies.items():
  idf[word]=math.log((1+N )/ (1+df_t)) # logatrima dari (total dokumen / total dokumen yang mengant

# Cek beberapa contoh nilai IDF
list(idf.items())[:10]

[('jakarta', 0.8663289956061171),
 ('kompascom', 2.3180061159888594),
 ('ponsel', 4.179724173823825),
 ('android', 5.117993812416755),
 ('huawei', 5.523458920524919),
 ('honor', 5.705780477318874),
 ('4c', 5.928924028633084),
 ('resmi', 2.519427844156233),
 ('meluncur', 4.830311739964974),
 ('pasar', 2.958509563063383)]

# TF-IDF

In [19]:
# Menghitung TF-IDF untuk setiap dokumen
def compute_tf_idf(word_freq, idf_dict):
  tf_idf = {}
  for word, tf in word_freq.items():
    if word in idf_dict:
      #TF-IDF = TF * IDF
      tf_idf[word] = tf * idf_dict[word]
  return tf_idf

tf_idf_list = []

for freq in df['word_frequency']:
  tf_idf_value = compute_tf_idf(freq, idf)
  tf_idf_list.append(tf_idf_value)

df['tf_idf'] = tf_idf_list
df.head()

,id,kategori,judul,isi,jumlahkata,clean,word_frequency,tf_idf
0,1,Teknologi,"Ponsel Huawei Honor 4C Dibanderol Rp 2,2 Juta","JAKARTA, KOMPAS.com Ponsel Android Huawei Hon...",315,jakarta kompascom ponsel android huawei honor ...,"{'jakarta': 1, 'kompascom': 1, 'ponsel': 5, 'a...","{'jakarta': 0.8663289956061171, 'kompascom': 2..."
1,2,Teknologi,Asosiasi: RPP E-commerce Tidak Sesuai Hasil Di...,"JAKARTA, KOMPAS.com - Sejak 2013, wacana tenta...",419,jakarta kompascom 2013 wacana rancangan peratu...,"{'jakarta': 2, 'kompascom': 1, '2013': 1, 'wac...","{'jakarta': 1.7326579912122342, 'kompascom': 2..."
2,3,Teknologi,"Pemesan ""iPhone Jadi Sabun"" Karyawan Pesaing L...","JAKARTA, KOMPAS.com Danis Darusman, pelanggan ...",265,jakarta kompascom danis darusman pelanggan laz...,"{'jakarta': 1, 'kompascom': 1, 'danis': 12, 'd...","{'jakarta': 0.8663289956061171, 'kompascom': 2..."
3,4,Teknologi,"""Autofeather Failure"", Momok bagi Pesawat Bali...",KOMPAS.com Salah satu momok yang dihadapi dala...,481,kompascom salah momok dihadapi pesawat turbopr...,"{'kompascom': 1, 'salah': 2, 'momok': 1, 'diha...","{'kompascom': 2.3180061159888594, 'salah': 3.1..."
4,5,Teknologi,Laptop Bezel Tertipis di Dunia Masuk Indonesia,"JAKARTA, KOMPAS.com - Resmi diperkenalkan pada...",227,jakarta kompascom resmi diperkenalkan ajang co...,"{'jakarta': 2, 'kompascom': 1, 'resmi': 1, 'di...","{'jakarta': 1.7326579912122342, 'kompascom': 2..."


In [20]:
# Pilih dokumen berdasarkan id berita
id_berita = 1 # ubah sesuai id berita yang ingin ditampilkan
row = df[df['id'] == id_berita].iloc[0]

# Ambil data TF dan TF-IDF dari dokumen tersebut
tf_data = row['word_frequency']
tfidf_data = row['tf_idf']

rows = []
for term, tf in tf_data.items():
  idf_value = idf.get(term, 0)
  tfidf_value = tfidf_data.get(term, 0)
  rows.append({
      'term': term,
      'tf': tf,
      'idf': round(idf_value, 4),
      'tfidf': round(tfidf_value, 4)
  })

  df_tfidf_detail = pd.DataFrame(rows)

df_tfidf_detail = df_tfidf_detail.sort_values(by='tfidf', ascending=False)

print(f"Hasil TF-IDF untuk id berita = {id_berita}\n")
print(df_tfidf_detail.to_string(index=False))

Hasil TF-IDF untuk id berita = 1

             term  tf    idf   tfidf
               4c   5 5.9289 29.6446
            honor   4 5.7058 22.8231
           ponsel   5 4.1797 20.8986
           darren   3 6.6221 19.8662
           huawei   3 5.5235 16.5704
         prosesor   3 4.7503 14.2508
     merancangnya   2 6.6221 13.2441
            photo   2 6.2166 12.4332
          refocus   2 5.9289 11.8578
         snapshot   2 5.9289 11.8578
       megapiksel   2 5.9289 11.8578
     penjualannya   2 5.9289 11.8578
            ultra   2 5.7058 11.4116
             fast   2 5.7058 11.4116
           selfie   2 5.7058 11.4116
         pengguna   3 3.7889 11.3666
               gb   2 5.5235 11.0469
             mode   2 5.3693 10.7386
         memotret   2 5.2358 10.4716
          android   2 5.1180 10.2360
           kamera   2 4.3708  8.7416
            fitur   2 4.0963  8.1927
             foto   2 4.0571  8.1142
      pembuatanya   1 6.6221  6.6221
        hisilicon   1 6.6221  6.6221
    

# Rangked Retrieval

In [21]:
# Mengumpulkan kata dalam setiap dokumen
all_terms_set = set() #Menyimpan kedalam bentuk set

for _, row in df.iterrows():
  # mengambil kata TF-IDF dokumen
  terms_in_doc = row['tf_idf'].keys()

  # menambahkan ke kumpulan semua kata
  all_terms_set.update(terms_in_doc)

all_terms = sorted(all_terms_set) #mengurutkan

In [23]:
all_terms[:10]

['0', '00', '00004', '001', '002', '003', '00304', '004', '005', '006']

In [22]:
# Membuat matrix tfidf menggunakan numpy
print("\nMembentuk matriks TF-IDF untuk seluruh dokumen dokumen")
doc_ids = df['id'].astype(int).tolist()

doc_vectors = np.array([
    [row['tf_idf'].get(term, 0) for term in all_terms]
    for _, row in tqdm(df.iterrows(), total=len(df))
])


Membentuk matriks TF-IDF untuk seluruh dokumen dokumen


100%|██████████| 1502/1502 [01:26<00:00, 17.42it/s]


In [26]:
# membuat tabel tf-idf
print("\nMembuat tabel TF-IDF")
df_tfidf_table = pd.DataFrame(doc_vectors.T, index=all_terms, columns=[f"d{doc_id}" for doc_id in doc_ids])
df_tfidf_table.index.name = "Term"


Membuat tabel TF-IDF


In [27]:
from IPython.display import display
display(df_tfidf_table.iloc[10200:10300])

,d1,d2,d3,d4,d5,d6,d7,d8,d9,d10,...,d1493,d1494,d1495,d1496,d1497,d1498,d1499,d1500,d1501,d1502
Term,,,,,,,,,,,,,,,,,,,,,
kandidat,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kandung,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kandungadik,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kandungan,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kandungnya,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
kartikawati,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kartina,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
kartoharjo,0.0,0.0,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [29]:
# 1. Pilih kolom 'di dari DataFrame
series_d1 =df_tfidf_table['d1']

# 2. Terapkan boolean Indexing: Filter series d1 untuk nilai yang lebih besar dari 0.0
nilai_diatas_nol_d1 = series_d1[series_d1 > 0.0]

# 3. Tampilkan hasilnya
print("--- Bobot TF-IDF (d1) di Atas 0.0 ---")
display(nilai_diatas_nol_d1)

--- Bobot TF-IDF (d1) di Atas 0.0 ---


,d1
Term,
10,2.651779
12,2.596720
13,3.731699
172015,2.062945
1835,6.216606
...,...
usb,6.622071
usia,4.019382
utamanya,4.830312


In [30]:
def preprocess_query(query):
  query = caseFolding(query)
  query = punctuationRemoval(query)
  query = remove_stopwords(query)
  return query

In [31]:
query = "ekonomi indonesia pertumbuhan inflasi"

In [33]:
query = preprocess_query(query)

In [34]:
q_tf = compute_word_frequency(query)
print("TF Query:", q_tf)

TF Query: {'ekonomi': 1, 'indonesia': 1, 'pertumbuhan': 1, 'inflasi': 1}


In [35]:
q_tfidf = compute_tf_idf(q_tf, idf)
print("TF-IDF Query:", q_tfidf)

TF-IDF Query: {'ekonomi': 3.0957106845768676, 'indonesia': 1.3990163271455394, 'pertumbuhan': 3.914021008090819, 'inflasi': 4.2241759363946585}


In [36]:
# Membuat vektor query dengan panjang yang sama seperti vektor dokumen
query_vector = np.array([q_tfidf.get(term, 0) for term in all_terms])

print("Panjang vector query:", len(query_vector))

Panjang vector query: 24247


In [37]:
# membuat fungsi untuk cosine similarity
def cosine_similarity_manual(vec1, vec2):
  dot_product = np.dot(vec1, vec2)
  norm_a = np.linalg.norm(vec1)
  norm_b = np.linalg.norm(vec2)
  if norm_a == 0 or norm_b == 0:
      return 0
  return dot_product / (norm_a * norm_b)

In [39]:
similarities = []

for i, doc_vec in enumerate(doc_vectors):
    sim = cosine_similarity_manual(query_vector, doc_vec)
    similarities.append((doc_ids[i], sim))

# Urutkan dari similarity tertinggi
ranked_results = sorted(similarities, key=lambda x: x[1], reverse=True)

# Tampilkan 10 dokumen teratas
print("\n === Hasil Ranked Retrieval (Top 10) === ")
for doc_id, sim in ranked_results[:10]:
    print(f"Doc {doc_id} -> similarity = {sim:.4f}")


 === Hasil Ranked Retrieval (Top 10) === 
Doc 396 -> similarity = 0.4679
Doc 402 -> similarity = 0.4239
Doc 881 -> similarity = 0.3382
Doc 43 -> similarity = 0.3266
Doc 446 -> similarity = 0.3075
Doc 1056 -> similarity = 0.2891
Doc 1285 -> similarity = 0.2822
Doc 871 -> similarity = 0.2243
Doc 414 -> similarity = 0.2224
Doc 502 -> similarity = 0.2221


In [40]:
doc_to_check = 396
row = df[df['id'] == doc_to_check]

if row.empty:
  print("ID dokumen tidak ditemukan.")
else:
  print("\n=== Isi Berita (ID:", doc_to_check, ") ===")
  print(row['isi'].iloc[0])


=== Isi Berita (ID: 396 ) ===
REPUBLIKA.CO.ID, JAKARTA -- Bank Indonesia memprediksi laju inflasi pada akhir 2015 akan mencapai level 4,2 persen hingga 4,3 persen secara tahunan meskipun saat ini masih di atas tujuh persen."Kalau kita kihat akhir kuartal I dan II 2015 ini inflasinya di atas tujuh persen, namun akhir tahun akan ada di plus minus satu persen year on year (secara tahunan)," kata Gubernur BI Agus Martowardojo, Rabu (1/7). Inflasi Juni 2015, sebagaimana baru dirilis oleh Badan Pusat Statistik (BPS), tercatat sebesar 0,54 persen (mtm). Tingkat inflasi tahun kalender (Januari-Juni) 2015 sebesar 0,96 persen dan tingkat inflasi tahun ke tahun (Juni 2015 terhadap Juni 2014) sebesar 7,26 persen.Menurut Agus, memang terdapat indikasi penurunan daya beli masyarakat selama enam bulan pertama tahun ini."Pengaruh daya beli agak lemah mungkin ada. Kita lihat, betul-betul ekonomi 2015 ini dijaga di semester kedua," kata Agus.Sementara itu, terkait inflasi Juni, Deputi Gubernur BI Perry